# 01 · Puente: matrices → estadística → economía
Este notebook conecta el Excel de absorción con las transformaciones que luego consumen econometría y ML: identidades, diferencias, medias móviles, z-scores, estacionalidad y factores latentes.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'replica_cygnus').exists())
sys.path.insert(0, str(ROOT/'src'))
from replica_cygnus.economic_intelligence import install_feature_mart, load_monthly_panel
install_feature_mart(); panel = load_monthly_panel()
PROJECT = panel['proyecto'].dropna().iloc[0]  # cambia aquí el proyecto a fiscalizar
d = panel.loc[panel['proyecto'].eq(PROJECT)].sort_values('periodo_mes').copy()
PROJECT, d.shape

In [ ]:
cols = ['periodo_mes','stock_total_departamentos_actual_ref','stock_inicio_observado','altas_mes','altas_acumuladas_ledger','stock_ofertado_observado_acum','separaciones_brutas_mes','caidas_mes','movimiento_neto_mes','ventas_minutas_mes','saldo_final_observado','absorcion_neta_mes','absorcion_stock_acumulada_observada']
d[cols].tail(24)

## La identidad que fiscaliza la matriz
Para stock disponible observado: `saldo_final = stock_inicio + altas - separaciones + caídas`. Una minuta no vuelve a retirar stock: la unidad ya salió de disponible al separarse.

In [ ]:
numeric_identity = ['stock_inicio_observado','altas_mes','separaciones_brutas_mes','caidas_mes','saldo_final_observado']
d[numeric_identity] = d[numeric_identity].apply(pd.to_numeric, errors='coerce')
d['saldo_reconstruido'] = d['stock_inicio_observado'] + d['altas_mes'] - d['separaciones_brutas_mes'] + d['caidas_mes']
d['gap_identidad'] = d['saldo_final_observado'] - d['saldo_reconstruido']
d[['periodo_mes','saldo_final_observado','saldo_reconstruido','gap_identidad']].tail(24)

## Transformaciones estadísticas
- **Diferencia:** cambio de nivel entre meses.
- **Media móvil:** señal de tendencia.
- **Z-score:** distancia respecto al comportamiento histórico del proyecto.
- **Log1p:** comprime colas de conteos sin perder ceros.
- **Sin/Cos:** estacionalidad circular (diciembre está cerca de enero).

Antes de transformar se normalizan los tipos numéricos provenientes de PostgreSQL (`Decimal`/NULL) a `float`, para que `diff`, `rolling` y NumPy operen de forma estable.

In [ ]:
numeric_stats = ['absorcion_neta_mes','movimiento_neto_mes','stock_inicio_observado']
d[numeric_stats] = d[numeric_stats].apply(pd.to_numeric, errors='coerce')
d['delta_abs'] = d['absorcion_neta_mes'].diff()
d['mov_ma3'] = d['movimiento_neto_mes'].rolling(3, min_periods=1).mean()
mu, sd = d['movimiento_neto_mes'].mean(), d['movimiento_neto_mes'].std(ddof=0)
d['mov_z'] = (d['movimiento_neto_mes']-mu)/(sd if pd.notna(sd) and sd != 0 else np.nan)
d['log_stock'] = np.log1p(d['stock_inicio_observado'].clip(lower=0))
d[['periodo_mes','movimiento_neto_mes','mov_ma3','mov_z','delta_abs','log_stock']].tail(18)

In [ ]:
plot_cols = ['stock_inicio_observado','saldo_final_observado','stock_ofertado_observado_acum','stock_total_departamentos_actual_ref']
d[plot_cols] = d[plot_cols].apply(pd.to_numeric, errors='coerce')
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(d['periodo_mes'], d['stock_inicio_observado'], label='Stock inicio observado')
ax.plot(d['periodo_mes'], d['saldo_final_observado'], label='Saldo final observado')
ax.plot(d['periodo_mes'], d['stock_ofertado_observado_acum'], label='Stock ofertado acumulado ledger')
ref = d['stock_total_departamentos_actual_ref'].max()
if pd.notna(ref): ax.axhline(ref, linestyle='--', label='Universo actual completo (ref)')
ax.set_title(f'{PROJECT} · Puente stock observado vs universo completo actual')
ax.legend(); ax.grid(alpha=.2); plt.show()

### Interpretación ejecutiva
La distancia entre `stock_ofertado_observado_acum` y `stock_total_departamentos_actual_ref` es **cobertura de evidencia**, no stock faltante necesariamente. Es una alerta de provenance: puede haber unidades que existan hoy pero cuya fecha de alta histórica nunca fue observada.